# Analyse de sentiments - API Microsoft Azure


### Table des matières

* [**1. Introduction**](#chapter1)
* [**2. Import des données**](#chapter2)
* [**3. Définition des fonctions**](#chapter3)
* [**4. Sample**](#chapter4)
* [**5. Preprocessing**](#chapter5)
* [**6. Test de l'API sans preprocessing**](#chapter6)    
* [**7. Test de l'API avec preprocessing**](#chapter7)    
* [**8. Test de l'API avec preprocessing et stemmatisation**](#chapter8)    
* [**9. Test de l'API avec preprocessing et lemmatisation**](#chapter9)    
* [**10. Synthèse des résultats**](#chapter10)    
* [**11. Aperçu des erreurs**](#chapter11)    
* [**12. Conclusion**](#chapter12)    
    

## 1 - Introduction <a class="anchor" id="chapter1"></a>

Dans ce notebook, l'API de microsoft Azure Cognitive Services pour le langage est testée pour l'analyse de sentiments.

## 2 - Import des données <a class="anchor" id="chapter2"></a>

Import des données préalablement nettoyées lors de l'analyse exploratoire

In [5]:
import pandas as pd

data_path = "./sentiment140/"
datas = pd.read_csv(data_path + "sentiment_datas_cleaned.csv", delimiter=',', encoding = "ISO-8859-1")


## 3 - Définition des fonctions <a class="anchor" id="chapter3"></a>

In [11]:
import os, requests, json
import sys, getopt
import time
import numpy as np
import sklearn
from sklearn.metrics import accuracy_score
from dotenv import load_dotenv


# get end point and subscription key from environment
def get_env_var():
   
    load_dotenv()

    if os.getenv('SENTIMENT_RSC_SUBSCRIPTION_KEY'):
        subscription_key = os.environ.get("SENTIMENT_RSC_SUBSCRIPTION_KEY")
    else:
        raise Exception('Please set/export the environment variable: {}'.format('SENTIMENT_RSC_SUBSCRIPTION_KEY'))

    if os.getenv('SENTIMENT_RSC_ENDPOINT'):
        endpoint = os.environ.get("SENTIMENT_RSC_ENDPOINT")
    else:
        raise Exception('Please set/export the environment variable: {}'.format('SENTIMENT_RSC_ENDPOINT'))

    return [subscription_key, endpoint]


# send a request to azure sentiment analysis
def sentiment(json_body):

    path = '/text/analytics/v3.2-preview.1/sentiment?opinionMining=false'
    constructed_url = endpoint + path

    headers = {
        'Ocp-Apim-Subscription-Key': subscription_key,
        'Content-type': 'application/json'
    }
    response = requests.post(constructed_url, headers=headers, json=json_body)
    return response.json()


# define the dataframe with the selected preprocess
def format_datas_to_azure(df,preprocess_type='clean_only'):

    text_clean_column = text_clean_column_dict[preprocess_type]
    
    prepare_datas = df[['id',text_clean_column]]
    prepare_datas = prepare_datas.rename(columns={text_clean_column: "text"})
    
    return prepare_datas


# return max score among positive and negative for scores within initial max not negative or positive 
def max_not_neutral(scores):
    
    not_neutral = {k: v for k, v in scores.items() if k in ['positive','negative']}
    return max(not_neutral, key = not_neutral.get)


# get appropriate positive or negative score for neutral scores in dataframe 
def treat_neutral_responses(df_sentiment):
    
    df_sentiment_neutral = df_sentiment.copy()
    df_sentiment_neutral = df_sentiment_neutral[~df_sentiment_neutral['sentiment'].isin(['positive','negative'])]
    df_sentiment_neutral['sentiment_treated'] = df_sentiment_neutral['confidenceScores'].apply(lambda x: max_not_neutral(x))
    df_sentiment = df_sentiment.merge(df_sentiment_neutral[['id','sentiment_treated']], on = 'id', how = 'left')
    df_sentiment['id'] = df_sentiment['id'].astype(str).astype(np.int64)
    df_sentiment['sentiment_treated'] = df_sentiment['sentiment_treated'].fillna(df_sentiment['sentiment'])
    
    return df_sentiment


# return a dataframe containing azure sentiment responses and catch error responses
def treat_sentiment_resp(sentiment_resp):
    
    df_sentiment_errors = pd.DataFrame()
    df_sentiment = pd.DataFrame()

    for result_type in sentiment_resp:
        if (result_type == 'documents'):
            df_sentiment = pd.DataFrame(sentiment_resp['documents'])
        elif (result_type == 'errors'):
            df_sentiment_errors = pd.DataFrame(sentiment_resp['errors'])

    if len(df_sentiment)>0:
        df_sentiment = treat_neutral_responses(df_sentiment)
        df_sentiment['label_azure'] = df_sentiment['sentiment_treated'].map(numeric_sentiment)
            
    return df_sentiment, df_sentiment_errors


# fill azure result dataframe with params results
def add_azure_results(azure_results, preprocess_type, accuracy, azure_sentiment_time):
    
    result={}    
    result['preprocess_type'] = preprocess_type
    result['accuracy'] = accuracy
    result['azure_sentiment_time'] = azure_sentiment_time
    azure_results = azure_results.append(result,ignore_index=True)
    return azure_results


# return a dataframe containing datas from start to end batch index
def prepare_batch(datas_to_send,start_batch,end_batch):  
    
    prepare_datas = datas_to_send.iloc[start_batch:end_batch,:]
    
    prepare_datas = prepare_datas.to_json(orient="records")
    prepare_datas = json.loads(prepare_datas)
    
    datas_to_send_batch = {}
    datas_to_send_batch['documents'] = prepare_datas   
    
    return datas_to_send_batch


# send batch datas to azure cognitive services 
def send_batchs_to_azure(datas_to_send, batch_size=10):

    start_time = time.time()
    
    data_batch_resp = {}
    len_df = len(datas_to_send)
    nb_batchs = len_df//batch_size
    last_batch_size = len_df%batch_size

    for batch in range(nb_batchs):
        start_batch = batch * batch_size
        end_batch = start_batch + batch_size
        datas_to_send_batch = prepare_batch(datas_to_send,start_batch,end_batch) 
        sentiment_resp = sentiment(datas_to_send_batch)
        data_batch_resp[batch] = sentiment_resp

    start_batch = nb_batchs * batch_size
    end_batch = start_batch + last_batch_size
    datas_to_send_batch = prepare_batch(datas_to_send,start_batch,end_batch) 
    sentiment_resp = sentiment(datas_to_send_batch)
    data_batch_resp[nb_batchs] = sentiment_resp
    
    azure_sentiment_time = time.time() - start_time

    return data_batch_resp, azure_sentiment_time


# aggregate batch response to general dataframe responses and errors
def treat_batch_resp(data_batch_resp):

    df_sentiments = pd.DataFrame()
    df_sentiments_errors = pd.DataFrame()
    
    for idx in data_batch_resp:
        df_sentiment, df_sentiment_errors = treat_sentiment_resp(data_batch_resp[idx])
        df_sentiments = pd.concat([df_sentiments,df_sentiment])
        df_sentiments_errors = pd.concat([df_sentiments_errors,df_sentiment_errors])

    if len(df_sentiments_errors)>0:
        print(f'Warning : {len(df_sentiments_errors)} tweet(s) without sentiment response. See df_sentiments_errors')

    return df_sentiments, df_sentiments_errors

## 4 - Sample <a class="anchor" id="chapter4"></a>

Extraction d'un échantillon de 1600 tweets

In [ ]:
def sample_datas(n):
    datas_0 = datas[datas['label']==0].sample(n//2)
    datas_4 = datas[datas['label']==4].sample(n//2)
    return pd.concat([datas_0,datas_4])


In [19]:
datas_sample = sample_datas(1600)


In [ ]:
# some initializations
text_clean_column_dict = {'without_clean':'text', 'clean_only':'text_clean', 'stem':'text_clean_stem','lem':'text_clean_lem'}
numeric_sentiment = {'negative':0, 'positive':4}
azure_results = pd.DataFrame(columns=['preprocess_type','accuracy','azure_sentiment_time'])
[subscription_key, endpoint] = get_env_var()


## 5 - Preprocessing <a class="anchor" id="chapter5"></a>

Application des fonctions de preprocessing définies dans le fichier preprocess_functions.py  
Création de 3 colonnes : nettoyage seul, avec stemmatisation, avec lemmatisation 

In [ ]:
# fonctions de preprocessing
%run -i ./preprocess_functions.py

In [20]:
datas_sample[text_clean_column_dict['clean_only']] = datas_sample['text'].apply(lambda x: preprocess(x, None))
datas_sample[text_clean_column_dict['stem']] = datas_sample['text'].apply(lambda x: preprocess(x, 'stem'))
datas_sample[text_clean_column_dict['lem']] = datas_sample['text'].apply(lambda x: preprocess(x, 'lem'))


## 6 - Test de l'API sans preprocessing <a class="anchor" id="chapter6"></a>

In [21]:
# preparation des données
datas_to_send = format_datas_to_azure(datas_sample, 'without_clean')

# envoi des données et traitement des réponses
data_batch_resp, azure_sentiment_time = send_batchs_to_azure(datas_to_send)
df_sentiments, df_sentiments_errors_no_clean = treat_batch_resp(data_batch_resp)

# merge des réponses
datas_with_azure_sentiment = datas_sample.merge(df_sentiments[['id','label_azure']], on = 'id')

# évaluation des réponses
acc_score = accuracy_score(datas_with_azure_sentiment['label'], datas_with_azure_sentiment['label_azure'])

# ajout des résultats
azure_results = add_azure_results(azure_results, 'without_clean', acc_score, azure_sentiment_time)


## 7 - Test de l'API avec preprocessing <a class="anchor" id="chapter7"></a>

In [22]:

# preparation des données
datas_to_send = format_datas_to_azure(datas_sample, 'clean_only')

# envoi des données et traitement des réponses
data_batch_resp, azure_sentiment_time = send_batchs_to_azure(datas_to_send)
df_sentiments, df_sentiments_errors_clean = treat_batch_resp(data_batch_resp)

# merge des réponses
datas_with_azure_sentiment = datas_sample.merge(df_sentiments[['id','label_azure']], on = 'id')

# évaluation des réponses
acc_score = accuracy_score(datas_with_azure_sentiment['label'], datas_with_azure_sentiment['label_azure'])

# ajout des résultats
azure_results = add_azure_results(azure_results, 'clean_only', acc_score, azure_sentiment_time)


## 8 - Test de l'API avec preprocessing et stemmatisation <a class="anchor" id="chapter8"></a>

In [27]:

# preparation des données
datas_to_send = format_datas_to_azure(datas_sample,'stem')

# envoi des données et traitement des réponses
data_batch_resp, azure_sentiment_time = send_batchs_to_azure(datas_to_send)
df_sentiments, df_sentiments_errors_stem = treat_batch_resp(data_batch_resp)

# merge des réponses
datas_with_azure_sentiment = datas_sample.merge(df_sentiments[['id','label_azure']], on = 'id')

# évaluation des réponses
acc_score = accuracy_score(datas_with_azure_sentiment['label'], datas_with_azure_sentiment['label_azure'])

# ajout des résultats
azure_results = add_azure_results(azure_results, 'stem', acc_score, azure_sentiment_time)


## 9 - Test de l'API avec preprocessing et lemmatisation <a class="anchor" id="chapter9"></a>

In [28]:

# preparation des données
datas_to_send = format_datas_to_azure(datas_sample,'lem')

# envoi des données et traitement des réponses
data_batch_resp, azure_sentiment_time = send_batchs_to_azure(datas_to_send)

# évaluation des réponses
df_sentiments, df_sentiments_errors_lem = treat_batch_resp(data_batch_resp)

# merge des réponses
datas_with_azure_sentiment = datas_sample.merge(df_sentiments[['id','label_azure']], on = 'id')
acc_score = accuracy_score(datas_with_azure_sentiment['label'], datas_with_azure_sentiment['label_azure'])

# ajout des résultats
azure_results = add_azure_results(azure_results, 'lem', acc_score, azure_sentiment_time)


## 10 - Synthèse des résultats <a class="anchor" id="chapter10"></a>

In [58]:
azure_results

,preprocess_type,accuracy,azure_sentiment_time
0,without_clean,0.742500,57.320783
1,clean_only,0.719045,61.546694
2,stem,0.675676,61.563812
3,lem,0.702074,60.604272


Le score le meilleur est avec le texte sans preprocessing.  
Le score de 74% est un bon score.

## 11 - Aperçu des erreurs <a class="anchor" id="chapter11"></a>

Les erreurs renvoyées par azure sentiment analysis sont dues à l'envoi de textes nulls à l'API.  
Ces textes nulls sont dus au fait que certains tweets ne possèdent aucune information analysable.  
Le preprocessing transforme ces tweets en valeur nulle.   

In [40]:
datas_sample[datas_sample['text_clean'] == '']

,label,id,text,text_clean,text_clean_stem,text_clean_lem
672501,0,2247188970,@QueenMiMiFan why??????,,,
482137,0,2179902518,And again http://twitpic.com/7h1fo,,,
888301,4,1687247892,@maybeedeluxe You too!,,,
1130387,4,1975692511,@mcraddictal,,,
1532119,4,2178187976,here i am again,,,
1158282,4,1979254473,@leahchu,,,
1472095,4,2065329820,@souljaboytellem,,,
1558884,4,2185942782,@suesues,,,
1295690,4,2003796603,@AndSheSpeaks ...,,,


## 12 - Conclusion <a class="anchor" id="chapter12"></a>

L'API est une bonne solution si on veut effectuer de l'analyse de sentiments avec une bonne performance sans investir dans le développement de modèles de machine learning et deep learning.
